In [25]:
import pandas as pd

#loading sheets
df_2009 = pd.read_excel('OneDrive/Desktop/FMCG_Project/data/online_retail_II.xlsx', sheet_name = 'Year 2009-2010')
df_2010 = pd.read_excel('OneDrive/Desktop/FMCG_Project/data/online_retail_II.xlsx', sheet_name = 'Year 2010-2011')

df = pd.concat([df_2009, df_2010], ignore_index=True)

print(f"Total rows: {len(df):,}")
print(f"Columns: {list(df.columns)}")

Total rows: 1,067,371
Columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


In [26]:
df.head(10)
df.info()
df.describe()
#check missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df)*100).round(2)
pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[ns]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 65.1+ MB


,Missing Count,Missing %
Invoice,0,0.00
StockCode,0,0.00
Description,4382,0.41
Quantity,0,0.00
InvoiceDate,0,0.00
Price,0,0.00
Customer ID,243007,22.77
Country,0,0.00


In [27]:
#Clean column names
df.columns = df.columns.str.strip().str.lower().str.replace(' ','_')
print(df.columns.tolist())

['invoice', 'stockcode', 'description', 'quantity', 'invoicedate', 'price', 'customer_id', 'country']


In [28]:
#InvoiceDate should be datetime
df['invoicedate'] = pd.to_datetime(df['invoicedate'])

#Extract useful time features (you'll need these for cohorent analysis)
df['year'] = df['invoicedate'].dt.year
df['month'] = df['invoicedate'].dt.month
df['year_month'] = df['invoicedate'].dt.to_period('M').astype(str)

print(df.dtypes)
print(df['year_month'].value_counts().sort_index().head(10))

invoice                object
stockcode              object
description            object
quantity                int64
invoicedate    datetime64[ns]
price                 float64
customer_id           float64
country                object
year                    int32
month                   int32
year_month             object
dtype: object
year_month
2009-12    45228
2010-01    31555
2010-02    29388
2010-03    41511
2010-04    34057
2010-05    35323
2010-06    39983
2010-07    33383
2010-08    33306
2010-09    42091
Name: count, dtype: int64


In [29]:
#Cancelled invoice start with 'C'
cancelled_mask = df['invoice'].astype(str).str.startswith('C')
print(f"Cancelled rows: {cancelled_mask.sum():,} ({cancelled_mask.mean()*100:.1f}%)")

df = df[~cancelled_mask].copy()
print(f"Rows after removing cancellations: {len(df):,}")

Cancelled rows: 19,494 (1.8%)
Rows after removing cancellations: 1,047,877


In [30]:
# Quantity and price should always be positive
print(f"Negative quantity rows: {(df['quantity']<=0).sum():,}")
print(f"Zero/Negative price rows: {(df['price']<=0).sum():,}")

df = df[(df['quantity']>0) & (df['price']>0)].copy()
print(f"Rows after quantity/price filter: {len(df):,}")

Negative quantity rows: 3,457
Zero/Negative price rows: 6,207
Rows after quantity/price filter: 1,041,670


In [31]:
print(f"Rows missing CustomerID: {df['customer_id'].isnull().sum():,}")

df = df.dropna(subset=['customer_id']).copy()

# CustomerID should be an integer
df['customer_id'] = df['customer_id'].astype(int)

print(f"Final rows after dropping null CustomerIDs: {len(df):,}")

Rows missing CustomerID: 236,121
Final rows after dropping null CustomerIDs: 805,549


In [32]:
# Revenue = quantity x unit price
df['revenue'] = df['quantity']*df['price']

print(f"Total revenue in dataset: €{df['revenue'].sum():,.2f}")
print(f"\nRevenue by country(top 10): ")
print(df.groupby('country')['revenue'].sum().sort_values(ascending=False).head(10))

Total revenue in dataset: €17,743,429.18

Revenue by country(top 10): 
country
United Kingdom    1.472315e+07
EIRE              6.216311e+05
Netherlands       5.542323e+05
Germany           4.312625e+05
France            3.552575e+05
Australia         1.699681e+05
Spain             1.091785e+05
Switzerland       1.003653e+05
Sweden            9.154972e+04
Denmark           6.986219e+04
Name: revenue, dtype: float64


In [33]:
print("===FINAL DATASET SUMMARY===")
print(f"Rows: {len(df):,}")
print(f"Unique customers: {df['customer_id'].nunique():,}")
print(f"Unique products: {df['stockcode'].nunique():,}")
print(f"Date range: {df['invoicedate'].min().date()} → {df['invoicedate'].max().date()}")
print(f"Countries: {df['country'].nunique()}")
print(f"Total revenue: £{df['revenue'].sum():,.2f}")

# Save
df.to_csv('OneDrive/Desktop/FMCG_Project/outputs/retail_clean.csv', index=False)
print("\n✓ Saved to outputs/retail_clean.csv")

===FINAL DATASET SUMMARY===
Rows: 805,549
Unique customers: 5,878
Unique products: 4,631
Date range: 2009-12-01 → 2011-12-09
Countries: 41
Total revenue: £17,743,429.18

✓ Saved to outputs/retail_clean.csv


In [34]:
df['invoicedate'] = pd.to_datetime(df['invoicedate'])
print(df['invoicedate'].dtype)  # should say datetime64[ns]

# Re-save
df.to_csv('OneDrive/Desktop/FMCG_Project/outputs/retail_clean.csv', index=False)
print("✓ Saved with fixed date type")

datetime64[ns]
✓ Saved with fixed date type
